# Data Preprocessing: A Nairobi Guide to Cleaning Data

In Machine Learning, **Preprocessing** is like preparing ingredients before cooking *Nyama Choma*. You don't just throw a whole goat on the fire; you skin it, chop it, and marinate it. 

If your data is "dirty" (different scales, missing values, or text), your model will struggle. We use these tools to make the data understandable for the computer.

### Agenda:
1. **Scaling**: Making sure all numbers are in the same neighborhood.
2. **Encoding**: Turning names/words into numbers (like Matatu routes).
3. **Imputation**: Filling in the gaps when data is missing.
4. **Text & Images**: Handling non-number data types.

### Step 0: Setting up our Tools
We need our basic tools like we need a good set of jikos and charcoal.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set(color_codes=True)
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

### 1. Generating Sample Data
First, let's create some dummy data to experiment with.

In [ ]:
df = pd.DataFrame({
    'x1': np.random.normal(0, 2, 10000),
    'x2': np.random.normal(5, 3, 10000),
    'x3': np.random.normal(-5, 5, 10000)
})

In [ ]:
df.plot.kde()

## 2. Scaling the Data

Imagine you are comparing the price of a **Matchbox (KES 5)** and a **Plot in Kitengela (KES 1,000,000)**. 

If you put these into a model, the large price of the plot will "bully" the matchbox price. But they are both important! **Scaling** brings them to a level playing field so the computer treats them fairly.

### A. StandardScaler (Like comparing fuel prices)

**Layman Terms:** We find the average (e.g., average price of Petrol in Nairobi) and see how much each station's price deviates from that average. 

**Logic from Source:** Calculate by subtracting mean of column & div by standard deviation.

**Mathematical Formula:**
$$z = \frac{x - \mu}{\sigma}$$
Where:
- $\mu$ is the mean (average).
- $\sigma$ is the standard deviation (spread).

**Nairobi Example:** If the average petrol price is 200 KES, a station selling at 205 KES is "above average". Standard scaler centers everything around 0 (the average).

In [ ]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
data_tf = ss.fit_transform(df)

### 🔬 Manual Demonstration: StandardScaler
Let's see the math in action using NumPy/Pandas!

In [ ]:
# Manual calculation using the formula: (x - mean) / std
manual_scaled = (df['x1'] - df['x1'].mean()) / df['x1'].std()

print("Scikit-learn result (first 5):", data_tf[:5, 0])
print("Manual result (first 5):      ", manual_scaled.values[:5])

In [ ]:
df_scaled = pd.DataFrame(data_tf, columns=['x1','x2','x3'])
df_scaled.plot.kde()

### B. MinMaxScaler (Like building heights in Upper Hill)

**Layman Terms:** We take the smallest value and call it 0, and the largest value and call it 1. Everything else falls in between.

**Logic from Source:** Calculate by subtracting min of column & div by difference between max & min.

**Mathematical Formula:**
$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

**Nairobi Example:** Imagine scaling building heights. **Upper Hill Kiosk** = 0. **Britam Tower** = 1. The **KCB Plaza** might be 0.7. Everything is now between 0 and 1.

In [ ]:
df = pd.DataFrame({
    # positive skew
    'x1': np.random.chisquare(8, 1000),
    # negative skew 
    'x2': np.random.beta(8, 2, 1000) * 40,
    # no skew
    'x3': np.random.normal(50, 3, 1000)
})
df.plot.kde()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
mm = MinMaxScaler()
data_tf = mm.fit_transform(df)

### 🔬 Manual Demonstration: MinMaxScaler
Let's calculate the range scale manually.

In [ ]:
# Manual calculation: (x - min) / (max - min)
min_val = df['x1'].min()
max_val = df['x1'].max()
manual_mm_scaled = (df['x1'] - min_val) / (max_val - min_val)

print("Scikit-learn result (first 5):", data_tf[:5, 0])
print("Manual result (first 5):      ", manual_mm_scaled.values[:5])

In [ ]:
pd.DataFrame(data_tf,columns=['x1','x2','x3']).plot.kde()

### C. RobustScaler (Smart Shopping in Gikomba)

**Layman Terms:** Sometimes you have "outliers" (extreme values). RobustScaler uses the median and the range between the 25th and 75th percentiles (IQR) so it doesn't get distracted by the extremes.

**Logic from Source:** Calculate by subtracting 1st-quartile & div by difference between 3rd-quartile & 1st-quartile.

**Visual Formula from Source:**
![RobustScaler Formula](https://github.com/awantik/machine-learning-slides/blob/master/pp2.PNG?raw=true)

**Mathematical Formula:**
$$x_{scaled} = \frac{x - Q_2(x)}{Q_3(x) - Q_1(x)}$$
- $Q_2$: Median
- $Q_3 - Q_1$: Interquartile Range (IQR)

**Nairobi Example:** If you are checking shirt prices in Gikomba. Most are 200 KES, but one "designer" shirt is 50,000 KES. RobustScaler ignores that 50k luxury shirt so it doesn't mess up your budget.

In [ ]:
df = pd.DataFrame({
    # Distribution with lower outliers
    'x1': np.concatenate([np.random.normal(20, 1, 1000), np.random.normal(1, 1, 25)]),
    # Distribution with higher outliers
    'x2': np.concatenate([np.random.normal(30, 1, 1000), np.random.normal(50, 1, 25)]),
})
df.plot.kde()

In [ ]:
from sklearn.preprocessing import RobustScaler
robustscaler = RobustScaler()
data_tf = robustscaler.fit_transform(df)

### Manual Demonstration: RobustScaler
Using Medians instead of Averages.

In [ ]:
# Manual calculation: (x - median) / (Q3 - Q1)
median = df['x1'].median()
q1 = df['x1'].quantile(0.25)
q3 = df['x1'].quantile(0.75)
manual_robust_scaled = (df['x1'] - median) / (q3 - q1)

print("Scikit-learn result (first 5):", data_tf[:5, 0])
print("Manual result (first 5):      ", manual_robust_scaled.values[:5])

In [ ]:
pd.DataFrame(data_tf, columns=['x1','x2']).plot.kde()

## 3. Normalization & Binarization
Sometimes we don't care about the *actual* value, just the magnitude or if it's above/below a certain "stage".

### Normalizer (Unit Vector)

**Logic from Source:** Each parameter value is obtained by dividing by magnitude.

**Visual Formula from Source:**
![Normalizer Formula](https://github.com/awantik/machine-learning-slides/blob/master/pp1.PNG?raw=true)

**Mathematical Logic:** Usually L2 norm:
$$\|x\|_2 = \sqrt{x_1^2 + x_2^2 + ... + x_n^2}$$
This makes every row lie on a circle (or sphere) with radius 1.

In [ ]:
df = pd.DataFrame({
    'x1': np.random.randint(-100, 100, 1000).astype(float),
    'y1': np.random.randint(-80, 80, 1000).astype(float),
    'z1': np.random.randint(-150, 150, 1000).astype(float),
})

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df.x1, df.y1, df.z1)

In [ ]:
from sklearn.preprocessing import Normalizer
normalizer = Normalizer()
data_tf = normalizer.fit_transform(df)

In [ ]:
df_norm = pd.DataFrame(data_tf, columns=['x1','y1','z1'])
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df_norm.x1, df_norm.y1, df_norm.z1)
plt.title("Post Normalization (Points on a Unit Sphere)")

### Binarizer (Switch: ON or OFF)

**Layman Terms:** Like a Boolean switch. Anything above 0 becomes 1, anything 0 or below becomes 0.

**Nairobi Example:** Are there Matatus at the Khoja stage? Yes (1) or No (0).

In [ ]:
X = np.array([[ 1., -1.,  2.],
              [ 2.,  0.,  0.],
              [ 0.,  1., -1.]])

In [ ]:
from sklearn.preprocessing import Binarizer
binarizer = Binarizer()
data_tf = binarizer.fit_transform(X)
data_tf

## 4. Encoding Categorical Data

Computers only understand numbers. They don't know what a "Matatu" or a "Boda Boda" is. We have to give them a numeric ID.

### A. Ordinal Encoding (Like Hotel Rankings)

**Layman Terms:** Use this when the categories have a clear order (Small < Medium < Large).

**Nairobi Example:** Ranking hotel prices in Westlands: **Budget** (1), **Mid-range** (2), **Luxury** (3). The order matters!

In [ ]:
df = pd.DataFrame({
    'Age':[33,44,22,44,55,22],
    'Income':['Low','Low','High','Medium','Medium','High']})
df['Income_encoded'] = df.Income.map({'Low':1,'Medium':2,'High':3})
df

### B. Nominal Encoding (One-Hot Encoding)

**Layman Terms:** Use this when there is no specific order. We create a special column for each category.

**Nairobi Example:** Types of transport: **Matatu**, **Boda Boda**, **Train**. A Boda isn't "more" than a Matatu; they are just different. We give each its own indicator.

In [ ]:
df = pd.DataFrame({
    'Age':[33,44,22,44,55,22],
    'Gender':['Male','Female','Male','Female','Male','Male']})
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
le = LabelEncoder()
df['gender_tf'] = le.fit_transform(df.Gender)
ohe = OneHotEncoder()
ohe.fit_transform(df[['gender_tf']]).toarray()

## 5. Imputation (Handling Missing Data)

**Layman Terms:** Sometimes data is missing (someone forgot to fill the form). Instead of throwing the whole record away, we fill it with a sensible guess, like the average.

**Nairobi Example:** If a bus manifest is missing the number of passengers for the 10:00 AM trip, we take the average number of passengers from the 9:00 AM and 11:00 AM trips and fill it in.

In [ ]:
df = pd.DataFrame({
    'A':[1,2,3,4,np.nan,7],
    'B':[3,4,1,np.nan,4,5]
})
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
imputer.fit_transform(df)

## 6. Polynomial Features

**Layman Terms:** Sometimes a straight line doesn't explain the relationship. We create deeper features (squares) to capture "curves" in the data.

**Mathematical Formula (Degree 2):**
$$(a, b) \implies (1, a, b, a^2, ab, b^2)$$

**Nairobi Example:** The relationship between rainfall and Maize yield in Kitale isn't a straight line. Too little rain is bad, enough is good, but too much flood is also bad. Polynomials help us map this curve.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
df = pd.DataFrame({'A':[1,2,3,4,5], 'B':[2,3,4,5,6]})
pol = PolynomialFeatures(degree=2)
pol.fit_transform(df)

## 7. Custom Transformers

**Layman Terms:** When the standard tools don't work, you build your own custom jiko.

**Nairobi Example:** Maybe you want to adjust all prices for the 16% VAT automatically. You write a function and tell the computer to apply it every time.

**Note:** Always use `.copy()` when modifying the dataframe inside a custom function to avoid changing your original data by mistake.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def mapping(x):
    # Bug fix: use .copy() to ensure original df isn't changed unintendedly
    res = x.copy()
    res['Age'] = res['Age']+2
    res['Counter'] = res['Counter'] * 2
    return res

customtransformer = FunctionTransformer(mapping, validate=False)
df = pd.DataFrame({
    'Age':[33,44,22,44,55,22],
    'Counter':[3,4,2,4,5,2],
     })
customtransformer.transform(df)

## 8. Text Processing (Vectorization)

**Layman Terms:** Machines can't read. We have to turn sentences into "word counts".

**Nairobi Example:** Analyzing Jumia Food reviews. We want to count how many times people mention "Fast Delivery" vs "Cold Food" in Kilimani.

In [ ]:
corpus = [
     'This is the first document awesome food.',
     'This is the second second document.',
     'And the third one the is mission impossible.',
     'Is this the first document?',
]
df = pd.DataFrame({'Text':corpus})
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
cv.fit_transform(df.Text).toarray()

### TF-IDF (The 'Keyword' Finder)

**Formula:** 
$$tf-idf(t,d) = tf(t,d) \times \log(\frac{N}{df(t)})$$

**Explanation:** It finds words that are frequent in one document but rare in others (like unique Swahili slang used only in certain routes).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')
tfs = vectorizer.fit_transform(df.Text)

# Bug fix: use get_feature_names_out() instead of get_feature_names()
print("Feature Names:", vectorizer.get_feature_names_out())
print("Sparse Matrix Shape:", tfs.shape)

## 9. Image Processing

**Layman Terms:** For a computer, an image is just a grid of numbers representing colors (Red, Green, Blue).

**Nairobi Example:** Traffic cameras on the Expressway take images. The computer sees these as numbers and identifies the car's plate number.

In [ ]:
from skimage.io import imread
import matplotlib.pyplot as plt

try:
    # Note: Using absolute path from source if available, or placeholder
    image = imread('./content/image.png')
    plt.imshow(image)
    plt.title("Visualizing as Numbers")
    plt.show()
except Exception as e:
    print(f'Error loading image: {e}. \nACTION: Please upload an image to ./content/image.png')